In [ ]:
!pip -q install -U openai gradio python-dotenv

In [ ]:
# imports
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY in env or .env"

In [ ]:
system_prompt = """
You are a senior Python engineer.
Your task: add docstrings and helpful inline comments to the given Python code.

Rules:
- Output ONLY valid Python code (no markdown, no backticks, no explanations).
- Do NOT change the behavior or logic of the code.
- Do NOT rename variables, functions, or classes unless absolutely required for clarity (prefer not to).
- Add docstrings for: module (optional), classes, public functions, and complex private helpers.
- Add short inline comments only where it improves readability (avoid commenting obvious lines).
- Keep docstrings concise but useful: purpose, args, returns, raises (if relevant).
""".strip()

In [ ]:
def strip_code_fences(text: str) -> str:
    text = (text or "").strip()
    if text.startswith("```"):
        parts = text.split("```")
        text = parts[1].strip() if len(parts) > 1 else text
        if text.lower().startswith("python"):
            text = text[6:].strip()
    return text

In [ ]:
def user_prompt_for(code: str, style: str) -> str:
    return f"""
Add docstrings and comments to the following code.

Docstring style: {style}

Constraints:
- Preserve exact runtime behavior.
- Keep output as a single Python file.

Python code:
{code}
""".strip()

In [ ]:
def add_docs(model: str, style: str, code: str):
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt_for(code, style)},
        ],
        temperature=0.2,
    )

    updated = strip_code_fences(resp.choices[0].message.content)
    return updated

In [ ]:
# Simple Gradio UI
DEFAULT_MODEL = "gpt-4o-mini" 

with gr.Blocks() as ui:
    gr.Markdown("# 📝 Docstring/Comment Generator (OpenAI)")

    with gr.Row():
        code_in = gr.Textbox(label="Python code (input)", lines=28, placeholder="Paste your Python module here...")
        code_out = gr.Textbox(label="Updated code (docstrings/comments added)", lines=28)

    with gr.Row():
        model = gr.Textbox(label="Model", value=DEFAULT_MODEL)
        style = gr.Dropdown(
            ["Google", "NumPy", "reST (Sphinx)"],
            value="Google",
            label="Docstring Style"
        )
        btn = gr.Button("Add Docstrings/Comments ✅")

    btn.click(add_docs, inputs=[model, style, code_in], outputs=[code_out])

ui.launch(inbrowser=True)